# M10t: QLoRA fine-tune of the scorer (FT-6, FT-7)

Run on a **free T4** (Colab: *Runtime → Change runtime type → T4 GPU*; Kaggle: *Accelerator → GPU T4*).

Needed in the working folder (`FT_DIR`, default: the current folder), as in `jobs/finetune/`:
`train_config.yaml`, `data/train.jsonl`, `data/val.jsonl`, `data/split-report.json`.
`data/test.jsonl` is **not** needed and is never read here.

```
!git clone -b <branch with the data> https://github.com/Zain-Shykh/idea-validator.git
%cd idea-validator/jobs/finetune
```

Output (git-ignored, `outputs/`): the training log, the LoRA adapter and the merged 16-bit model.
Then follow `export_gguf.md`.

In [ ]:
import os
from pathlib import Path

import yaml

FT_DIR = Path(os.environ.get("FT_DIR", ".")).resolve()
OUT = FT_DIR / "outputs"
LOG_PATH = Path(os.environ.get("FT_LOG_PATH", OUT / "finetune-train-log.json"))
CONFIG = yaml.safe_load((FT_DIR / "train_config.yaml").read_text(encoding="utf-8"))
print(yaml.safe_dump(CONFIG, sort_keys=False))

In [ ]:
# the pinned versions of train_config.yaml. Restart the runtime if Colab asks for it.
packages = " ".join(CONFIG["packages"])
%pip install -q {packages}

In [ ]:
import hashlib
import json

report = json.loads((FT_DIR / "data/split-report.json").read_text(encoding="utf-8"))


def load_part(name):
    raw = (FT_DIR / f"data/{name}.jsonl").read_bytes()
    digest = hashlib.sha256(raw).hexdigest()
    # the files the leakage assertions passed on, and no others (FT-4)
    if digest != report["sha256"][name]:
        raise SystemExit(f"{name}.jsonl is not the file split.py wrote: run split.py again")
    rows = [json.loads(line) for line in raw.decode("utf-8").split("\n") if line.strip()]
    return rows, digest


parts = {name: load_part(name) for name in ("train", "val")}
print({name: len(rows) for name, (rows, _) in parts.items()}, "fake:", report["fake"])

In [ ]:
# unsloth first: it patches transformers and trl when imported
# isort: off
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
# isort: on

if not torch.cuda.is_available():
    raise SystemExit("no GPU: choose a T4 runtime")
GPU = torch.cuda.get_device_name(0)
print(GPU)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
)
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora"]["r"],
    lora_alpha=CONFIG["lora"]["alpha"],
    lora_dropout=CONFIG["lora"]["dropout"],
    target_modules=CONFIG["lora"]["target_modules"],
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["training"]["seed"],
)

In [ ]:
# the base model's own chat template: what llama.cpp applies when it serves the model
def render(rows):
    texts, too_long = [], 0
    for row in rows:
        text = tokenizer.apply_chat_template(
            row["messages"], tokenize=False, add_generation_prompt=False
        )
        # a cut row would lose its label, so it is left out instead
        tokens = tokenizer(text=text, add_special_tokens=False)["input_ids"]
        if len(tokens) > CONFIG["max_seq_length"]:
            too_long += 1
        else:
            texts.append({"text": text})
    return texts, too_long


rendered = {name: render(rows) for name, (rows, _) in parts.items()}
for name, (texts, too_long) in rendered.items():
    print(f"{name}: {len(texts)} rows, {too_long} longer than {CONFIG['max_seq_length']} tokens")
    if not texts:
        raise SystemExit(f"{name}: no row fits; raise max_seq_length in train_config.yaml")

In [ ]:
settings = CONFIG["training"]
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=Dataset.from_list(rendered["train"][0]),
    eval_dataset=Dataset.from_list(rendered["val"][0]),
    args=SFTConfig(
        output_dir=str(OUT / "checkpoints"),
        dataset_text_field="text",
        max_length=CONFIG["max_seq_length"],
        learning_rate=settings["learning_rate"],
        num_train_epochs=settings["epochs"],
        per_device_train_batch_size=settings["per_device_batch_size"],
        per_device_eval_batch_size=settings["per_device_batch_size"],
        gradient_accumulation_steps=settings["gradient_accumulation_steps"],
        logging_steps=settings["logging_steps"],
        eval_strategy="steps",
        eval_steps=settings["eval_steps"],
        save_strategy="no",
        optim=settings["optimizer"],
        fp16=settings["fp16"],
        bf16=False,
        seed=settings["seed"],
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part=CONFIG["chat_markers"]["instruction"],
    response_part=CONFIG["chat_markers"]["response"],
)

In [ ]:
from datetime import UTC, datetime
from importlib import metadata


def version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


started = datetime.now(UTC)
result = trainer.train()
trainer.evaluate()  # the validation loss after the last step
log = {
    "started_at": started.isoformat(),
    "finished_at": datetime.now(UTC).isoformat(),
    "gpu": GPU,
    "settings": CONFIG,
    "versions": {
        name: version(name)
        for name in ("unsloth", "trl", "transformers", "peft", "datasets", "bitsandbytes", "torch")
    },
    "data": {
        name: {
            "rows_in_file": len(rows),
            "rows_used": len(rendered[name][0]),
            "too_long": rendered[name][1],
            "sha256": digest,
        }
        for name, (rows, digest) in parts.items()
    },
    "fake_data": report["fake"],
    "metrics": result.metrics,
    # training "loss" and validation "eval_loss" by step: the two curves
    "log_history": trainer.state.log_history,
}
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.write_text(json.dumps(log, indent=2) + "\n", encoding="utf-8")
print("log written to", LOG_PATH)

In [ ]:
# watch validation loss (FT-6): if it rises while training loss falls, lower epochs and rerun
for entry in trainer.state.log_history:
    shown = {k: entry[k] for k in ("step", "epoch", "loss", "eval_loss") if k in entry}
    if "loss" in shown or "eval_loss" in shown:
        print(shown)

In [ ]:
model.save_pretrained(str(OUT / "adapter"))
tokenizer.save_pretrained(str(OUT / "adapter"))
# adapter merged into the base weights, 16 bit: the input of the GGUF conversion (export_gguf.md)
model.save_pretrained_merged(str(OUT / "merged"), tokenizer, save_method="merged_16bit")